# Configuration

Quick dependency check before running the code. 

In [1]:
import importlib.util

required_modules = ["torch", "numpy", "pandas", "plotly"]
pip_name_map = {"sklearn": "scikit-learn", "cv2": "opencv-python", "PIL": "Pillow", "yaml": "PyYAML"}

missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
missing_pip = [pip_name_map.get(m, m) for m in missing]

print("Dependency check")
if missing_pip:
    print("Missing packages:", ", ".join(missing_pip))
    print("Install:")
    print("  pip install " + " ".join(missing_pip))
    raise SystemExit("Install missing packages, then re-run this cell.")

print("All required packages are installed.")

Dependency check
All required packages are installed.


In [2]:
import os
import torch

# Output paths (read pre-trained models + write test/analysis results)
output_root_dir = "/Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output"
output_kfold_dir = os.path.join(output_root_dir, "training")

tests_root_dir = "/Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/Independent_tests"
batch_output_root = os.path.join(output_root_dir, "independent_test_batches")
html_output_dir = os.path.join(output_root_dir, "html_visualizations")
output_rank_dir = os.path.join(output_root_dir, "saliency_rank_curves")

for path in [batch_output_root, html_output_dir, output_rank_dir]:
    os.makedirs(path, exist_ok=True)

class_names = {'Discoide': 0, 'Levallois': 1, 'Laminaire': 2}
k_folds = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✓ Trained models dir : {output_kfold_dir}")
print(f"✓ Test sets dir      : {tests_root_dir}")
print(f"✓ Device             : {device}")

✓ Trained models dir : /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/training
✓ Test sets dir      : /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/Independent_tests
✓ Device             : cpu


# Independent Test Inference

In [3]:
import os
import importlib
from datetime import datetime

import pandas as pd
import pointnet2_inference as p2i

p2i = importlib.reload(p2i)
run_inference_on_mesh_folder = p2i.run_inference_on_mesh_folder

folder_true_class_map = {
    "GBP_Disc": "Discoide",
    "UoM_Disc": "Discoide",
    "Nubian": "Levallois",
    "UoM_Leva": "Levallois",
    "UoM_Blade": "Laminaire",
    "ModelNet": None,
}

# Run control
RUN_INDEPENDENT_TEST = True      # False -> skip this section
CHECK_ALREADY_RUN = True         # If True, skip tests already present in the main output folder
RERUN_EXISTING_TESTS = True     # True -> run again and save to a separate parallel rerun folder
RUN_TAG = "rep_v1"              # Used only when RERUN_EXISTING_TESTS=True; None -> timestamp

batch_rerun_root = os.path.join(output_root_dir, "independent_test_batches_reruns")


def has_npy_files(folder):
    if not os.path.isdir(folder):
        return False
    return any(f.lower().endswith('.npy') for f in os.listdir(folder))


def is_test_already_run(test_name, output_root):
    test_dir = os.path.join(output_root, test_name)
    return all(os.path.exists(os.path.join(test_dir, f)) for f in [
        "ensemble_predictions.csv",
        "all_model_predictions_detailed.csv",
    ])


analysis_batch_output_root = batch_output_root

if not RUN_INDEPENDENT_TEST:
    print("⏭️ Independent Test Inference skipped (RUN_INDEPENDENT_TEST=False)")
else:
    print(f"Scanning test directory: {tests_root_dir}\n")

    valid_tests = []
    missing_or_invalid = []
    for test_name in sorted(folder_true_class_map.keys()):
        test_path = os.path.join(tests_root_dir, test_name)
        if os.path.isdir(test_path) and has_npy_files(test_path):
            valid_tests.append(test_name)
        else:
            missing_or_invalid.append(test_name)

    already_run = []
    tests_to_run = []
    for test_name in valid_tests:
        if CHECK_ALREADY_RUN and is_test_already_run(test_name, batch_output_root):
            already_run.append(test_name)
        else:
            tests_to_run.append(test_name)

    if RERUN_EXISTING_TESTS:
        run_suffix = RUN_TAG or datetime.now().strftime("%Y%m%d_%H%M%S")
        analysis_batch_output_root = os.path.join(batch_rerun_root, f"rerun_{run_suffix}")
        os.makedirs(analysis_batch_output_root, exist_ok=True)
        tests_to_run = list(valid_tests)
        print("Mode: rerun existing tests")
        print(f"Rerun output root: {analysis_batch_output_root}")
    else:
        analysis_batch_output_root = batch_output_root
        print("Mode: standard run")
        print(f"Main output root: {analysis_batch_output_root}")
        print(f"Check existing outputs: {CHECK_ALREADY_RUN}")

    if tests_to_run:
        print(f"Tests to run: {tests_to_run}")
    if already_run and not RERUN_EXISTING_TESTS:
        print(f"Already completed (skipped): {already_run}")
    if missing_or_invalid:
        print(f"Missing or no NPY files: {missing_or_invalid}")
    if not tests_to_run and not already_run:
        print("No tests found.")

    for test_name in tests_to_run:
        input_dir = os.path.join(tests_root_dir, test_name)
        output_dir = os.path.join(analysis_batch_output_root, test_name)
        os.makedirs(output_dir, exist_ok=True)

        true_class = folder_true_class_map[test_name]
        label_dict_test = None
        if true_class is not None:
            label_dict_test = {
                os.path.splitext(name)[0]: true_class
                for name in os.listdir(input_dir)
                if name.lower().endswith('.npy')
            }

        print(f"\n🔄 {test_name} (expected: {true_class})")

        run_inference_on_mesh_folder(
            input_folder=input_dir,
            kfold_results_dir=output_kfold_dir,
            k_folds=k_folds,
            num_classes=len(class_names),
            class_names=class_names,
            device=device,
            output_dir=output_dir,
            npoints=1024,
            normalize=True,
            batch_size=32,
            label_dict=label_dict_test,
            seed=42,
            deterministic=True,
        )

        ensemble_path = os.path.join(output_dir, "ensemble_predictions.csv")
        if os.path.exists(ensemble_path):
            df = pd.read_csv(ensemble_path)
            print(f"   ✓ {len(df)} predictions saved")

    if not tests_to_run:
        print("\nℹ️ No new tests run.")
    else:
        print("\n✅ Inference complete.")
        print(f"Results root used for downstream analysis: {analysis_batch_output_root}")

Scanning test directory: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/Independent_tests

Mode: rerun existing tests
Rerun output root: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/independent_test_batches_reruns/rerun_rep_v1
Tests to run: ['GBP_Disc', 'ModelNet', 'Nubian', 'UoM_Blade', 'UoM_Disc', 'UoM_Leva']

🔄 GBP_Disc (expected: Discoide)
🔬 RUNNING POINTNET++ INFERENCE ON INDEPENDENT FILES
Input folder: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/Independent_tests/GBP_Disc
K-fold models from: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/training
Output directory: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/independent_test_batches_reruns/rerun_rep_v1/GBP_Disc
Total folds expected: 5
Supported formats: PLY, STL, NPY, OBJ
Deterministic mode: ON (seed=42)
✅ Created datase

Inference Fold 1: 100%|██████████| 1/1 [00:12<00:00, 12.04s/it]



📂 RUNNING INFERENCE WITH FOLD 2 MODEL


Inference Fold 2: 100%|██████████| 1/1 [00:09<00:00,  9.93s/it]



📂 RUNNING INFERENCE WITH FOLD 3 MODEL


...


📂 RUNNING INFERENCE WITH FOLD 4 MODEL


Inference Fold 4: 100%|██████████| 3/3 [00:18<00:00,  6.01s/it]



📂 RUNNING INFERENCE WITH FOLD 5 MODEL


Inference Fold 5: 100%|██████████| 3/3 [00:16<00:00,  5.51s/it]

✅ Combined predictions saved: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/independent_test_batches_reruns/rerun_rep_v1/UoM_Leva/all_model_predictions_detailed.csv
✅ Ensemble predictions saved: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/independent_test_batches_reruns/rerun_rep_v1/UoM_Leva/ensemble_predictions.csv
🎉 Inference complete!
   ✓ 68 predictions saved

✅ Inference complete.
Results root used for downstream analysis: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/independent_test_batches_reruns/rerun_rep_v1


# Critical Point Analysis

In [4]:
from pointnet2_critical_analysis import check_critical_point_status

critical_fraction = 0.10
saliency_folds = [1, 2, 3, 4, 5]
CRITICAL_RANDOM_SEED = 42
SKIP_EXISTING_CRITICAL = True
analysis_batch_output_root = globals().get('analysis_batch_output_root', batch_output_root)

status_df = check_critical_point_status(
    analysis_batch_output_root,
    tests_root_dir,
    folder_true_class_map.keys(),
    skip_existing_critical=SKIP_EXISTING_CRITICAL,
 )

print("Critical Point Analysis Status:")
print(status_df.to_string(index=False))
print(f"\nReady test sets: {status_df['ready'].sum()} / {len(status_df)}")
print(f"Critical seed: {CRITICAL_RANDOM_SEED}")
print(f"Analysis output root: {analysis_batch_output_root}")
if SKIP_EXISTING_CRITICAL:
    print(f"Pending files (incremental): {int(status_df['pending'].sum())}")

Critical Point Analysis Status:
 test_set      source_type                                                                                                source_dir  npy_count  existing_critical  pending  ready
 GBP_Disc dataset_root_npy  /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/Independent_tests/GBP_Disc         29                  0       29   True
 ModelNet dataset_root_npy  /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/Independent_tests/ModelNet         60                  0       60   True
   Nubian dataset_root_npy    /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/Independent_tests/Nubian        178                  0      178   True
UoM_Blade dataset_root_npy /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/Independent_tests/UoM_Blade         30                  0       30   True
 UoM_Disc dataset_root_npy  /Users/lili/Library/CloudStorage/Dropbox/Proj

In [5]:
from pointnet2_critical_analysis import compute_critical_points_for_tests

critical_fraction = globals().get('critical_fraction', 0.10)
saliency_folds = globals().get('saliency_folds', [1, 2, 3, 4, 5])
SKIP_EXISTING_CRITICAL = globals().get('SKIP_EXISTING_CRITICAL', True)
CRITICAL_RANDOM_SEED = globals().get('CRITICAL_RANDOM_SEED', 42)
analysis_batch_output_root = globals().get('analysis_batch_output_root', batch_output_root)

print(
    f"Using critical config → fraction={critical_fraction}, "
    f"folds={saliency_folds}, skip_existing={SKIP_EXISTING_CRITICAL}, seed={CRITICAL_RANDOM_SEED}"
 )
print(f"Analysis output root: {analysis_batch_output_root}")

test_results, critical_summary, models, available_folds, device = compute_critical_points_for_tests(
    batch_output_root=analysis_batch_output_root,
    tests_root_dir=tests_root_dir,
    test_names=folder_true_class_map.keys(),
    output_kfold_dir=output_kfold_dir,
    saliency_folds=saliency_folds,
    critical_fraction=critical_fraction,
    skip_existing_critical=SKIP_EXISTING_CRITICAL,
    num_classes=3,
    random_seed=CRITICAL_RANDOM_SEED,
    device=None,
 )

print("\n✅ Critical point analysis complete!")
print(
    f"Total processed: {critical_summary['total_processed']} | "
    f"skipped existing: {critical_summary['total_skipped']} | "
    f"errors: {critical_summary['total_errors']}"
 )
for test_name, result in test_results.items():
    print(
        f"  {test_name}: source={result['source_type']}, total={result['n_files']}, "
        f"processed={result['processed']}, skipped={result['skipped_existing']}, errors={result['errors']}"
    )

Using critical config → fraction=0.1, folds=[1, 2, 3, 4, 5], skip_existing=True, seed=42
Analysis output root: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/independent_test_batches_reruns/rerun_rep_v1
Loading ensemble models...
✓ Loaded 5 models from folds: [1, 2, 3, 4, 5]


Processing: GBP_Disc | source=dataset_root_npy | total=29 | pending=29


Computing saliency - GBP_Disc: 100%|██████████| 29/29 [00:38<00:00,  1.33s/it]



Processing: ModelNet | source=dataset_root_npy | total=60 | pending=60


Computing saliency - ModelNet: 100%|██████████| 60/60 [01:16<00:00,  1.27s/it]



Processing: Nubian | source=dataset_root_npy | total=178 | pending=178


...


Processing: UoM_Disc | source=dataset_root_npy | total=56 | pending=56


Computing saliency - UoM_Disc: 100%|██████████| 56/56 [01:10<00:00,  1.26s/it]



Processing: UoM_Leva | source=dataset_root_npy | total=68 | pending=68


Computing saliency - UoM_Leva: 100%|██████████| 68/68 [01:26<00:00,  1.28s/it]


✅ Critical point analysis complete!
Total processed: 421 | skipped existing: 0 | errors: 0
  GBP_Disc: source=dataset_root_npy, total=29, processed=29, skipped=0, errors=0
  ModelNet: source=dataset_root_npy, total=60, processed=60, skipped=0, errors=0
  Nubian: source=dataset_root_npy, total=178, processed=178, skipped=0, errors=0
  UoM_Blade: source=dataset_root_npy, total=30, processed=30, skipped=0, errors=0
  UoM_Disc: source=dataset_root_npy, total=56, processed=56, skipped=0, errors=0
  UoM_Leva: source=dataset_root_npy, total=68, processed=68, skipped=0, errors=0


In [6]:
import os
import importlib
import plotly.io as pio
import pointnet2_critical_viz as critical_viz
from pointnet2_critical_analysis import resolve_npy_source_dir

critical_viz = importlib.reload(critical_viz)
build_critical_overlay_figure = critical_viz.build_critical_overlay_figure
analysis_batch_output_root = globals().get('analysis_batch_output_root', batch_output_root)

os.makedirs(html_output_dir, exist_ok=True)

print("Generating critical point visualizations...\n")
print(f"Analysis output root: {analysis_batch_output_root}")

for test_name in sorted(test_results.keys()):
    test_batch_dir = os.path.join(analysis_batch_output_root, test_name)
    test_info = test_results.get(test_name, {})
    source_npy_dir = test_info.get('source_npy_dir')

    if not source_npy_dir:
        source_npy_dir, _, _ = resolve_npy_source_dir(analysis_batch_output_root, tests_root_dir, test_name)

    critical_points_dir = os.path.join(test_batch_dir, 'critical_points')
    ensemble_csv = os.path.join(test_batch_dir, 'ensemble_predictions.csv')

    if not all(os.path.exists(p) for p in [source_npy_dir, critical_points_dir, ensemble_csv]):
        print(f"⏭️  {test_name}: Missing required files")
        continue
    
    fig_grid, n_flakes = build_critical_overlay_figure(
        test_name=test_name,
        ensemble_predictions_csv=ensemble_csv,
        source_npy_dir=source_npy_dir,
        critical_points_dir=critical_points_dir,
    )
    if fig_grid is None or n_flakes == 0:
        print(f"⏭️  {test_name}: No critical-point overlays available")
        continue
    
    html_path = os.path.join(html_output_dir, f"critical_points_visualization_pointnet2_{test_name}.html")
    pio.write_html(fig_grid, html_path)
    print(f"✓ {test_name}: {n_flakes} flakes → HTML")

print(f"\n✅ Visualization complete! Files saved to {html_output_dir}")

Generating critical point visualizations...

Analysis output root: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/independent_test_batches_reruns/rerun_rep_v1
✓ GBP_Disc: 29 flakes → HTML
✓ ModelNet: 60 flakes → HTML
✓ Nubian: 178 flakes → HTML
✓ UoM_Blade: 30 flakes → HTML
✓ UoM_Disc: 56 flakes → HTML
✓ UoM_Leva: 68 flakes → HTML

✅ Visualization complete! Files saved to /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/html_visualizations


# Saliency Rank Curves

Computes cumulative saliency rank curves for each independent test set and saves HTML outputs plus one compact summary CSV.

In [7]:
import os
import importlib
import numpy as np
import pandas as pd
import torch
import pointnet2_critical_viz as critical_viz
from pointnet2_critical_analysis import compute_rank_curves_for_test, resolve_npy_source_dir

critical_viz = importlib.reload(critical_viz)
plot_rank_curves = critical_viz.plot_rank_curves
analysis_batch_output_root = globals().get('analysis_batch_output_root', batch_output_root)

percentiles = [1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
os.makedirs(output_rank_dir, exist_ok=True)

if 'models' not in locals() or not models:
    from pointnet2_critical_analysis import load_ensemble_models
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    models, available_folds = load_ensemble_models(
        output_kfold_dir,
        saliency_folds,
        num_classes=3,
        device=device,
    )

print("Computing saliency rank curves...\n")
print(f"Analysis output root: {analysis_batch_output_root}")

if 'device' not in locals() or device is None:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

all_summary_rows = []

for test_name in sorted(folder_true_class_map.keys()):
    test_info = test_results.get(test_name, {}) if 'test_results' in globals() else {}
    source_npy_dir = test_info.get('source_npy_dir')

    if not source_npy_dir:
        source_npy_dir, _, _ = resolve_npy_source_dir(analysis_batch_output_root, tests_root_dir, test_name)

    if not source_npy_dir or not os.path.isdir(source_npy_dir):
        continue
    
    flake_curves = compute_rank_curves_for_test(
        test_name=test_name,
        source_npy_dir=source_npy_dir,
        models=models,
        percentiles=percentiles,
        device=device,
    )
    
    if not flake_curves:
        print(f"  {test_name}: No curves generated")
        continue
    
    for p in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
        idx = percentiles.index(p) if p in percentiles else len(percentiles) - 1
        values = [fc['curve'][idx] for fc in flake_curves]
        all_summary_rows.append({
            'test_set': test_name,
            'percentage': p,
            'mean_cumulative_saliency': float(np.mean(values)),
            'std_cumulative_saliency': float(np.std(values)),
            'n_flakes': len(values),
        })
    
    html_path = plot_rank_curves(test_name, flake_curves, percentiles, output_rank_dir)
    print(f"✓ {test_name}: {len(flake_curves)} curves → {os.path.basename(html_path)}")

if all_summary_rows:
    summary_df = pd.DataFrame(all_summary_rows)
    summary_csv = os.path.join(output_rank_dir, 'all_tests_saliency_rank_summary.csv')
    summary_df.to_csv(summary_csv, index=False)
    print(f"\n✅ Rank curves complete! Summary saved to {summary_csv} ({len(summary_df)} rows)")
else:
    print("\nℹ️  No rank curves generated. Check NPY availability.")

Computing saliency rank curves...

Analysis output root: /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/independent_test_batches_reruns/rerun_rep_v1


Computing rank curves - GBP_Disc: 100%|██████████| 29/29 [00:38<00:00,  1.33s/it]


✓ GBP_Disc: 29 curves → GBP_Disc_saliency_rank_curves.html


Computing rank curves - ModelNet: 100%|██████████| 60/60 [01:13<00:00,  1.22s/it]


✓ ModelNet: 60 curves → ModelNet_saliency_rank_curves.html


...

✓ UoM_Blade: 30 curves → UoM_Blade_saliency_rank_curves.html


Computing rank curves - UoM_Disc: 100%|██████████| 56/56 [01:10<00:00,  1.26s/it]


✓ UoM_Disc: 56 curves → UoM_Disc_saliency_rank_curves.html


Computing rank curves - UoM_Leva: 100%|██████████| 68/68 [01:16<00:00,  1.13s/it]


✓ UoM_Leva: 68 curves → UoM_Leva_saliency_rank_curves.html

✅ Rank curves complete! Summary saved to /Users/lili/Library/CloudStorage/Dropbox/Projects/Machine_Learning/Data_share/PointNet2/Output/saliency_rank_curves/all_tests_saliency_rank_summary.csv (60 rows)
